In [391]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"U:\data\HMC\HMC1\day13"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="Rat"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
theta_epochs = nel.EpochArray(
    state_dict["THETA"],
)
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']  # this is the field you're interested in
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    # Now you should have shape (N, 2)
    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")
n_pyr=len(E_spikes)
print(f"number of pyramidal cells: {n_pyr}")
print(f"nrem duration: {nrem_epochs.durations.sum():.2f} seconds")
print(f"rrem duration: {rem_epochs.durations.sum():.2f} seconds")
print(f"theta duration: {theta_epochs.durations.sum():.2f} seconds")

Folder already exists: U:\data\HMC\HMC1\day13\complex_system
number of cells: 105
number of pyramidal cells: 95
nrem duration: 11199.00 seconds
rrem duration: 1449.00 seconds
theta duration: 1101.00 seconds


In [332]:
epoch_df

,name,startTime,stopTime,environment,behavioralParadigm,manipulation,stimuli,notes,basepath
0,preSleep_210416_073406,0.0000,10570.13115,sleep,NaN,NaN,NaN,NaN,U:\data\HMC\HMC1\day13
1,maze_210416_130639,10570.1312,13029.35035,linear,6,NaN,NaN,NaN,U:\data\HMC\HMC1\day13
2,postSleep_210416_135513,13029.3504,24558.07355,sleep,NaN,NaN,NaN,NaN,U:\data\HMC\HMC1\day13


In [333]:
import numpy as np

# ---- 1) flatten across units ----
# spike_times: length-49 container, each element is 1D array of spike times (seconds)
all_spikes = np.sort(np.concatenate([
    np.asarray(st, float).ravel()
    for st in spike_times
    if st is not None and len(st) > 0
]))

# (optional) drop NaNs/inf
all_spikes = all_spikes[np.isfinite(all_spikes)]
def mean_isi_in_epochs(all_spikes, epochs):
    """
    all_spikes: 1D sorted spike times (seconds)
    epochs: array-like of shape (n_epochs, 2) with [start, end] in seconds
            (if you have nel.EpochArray, pass epochs.data)
    Returns:
      per_epoch_mean_isi (seconds), overall_mean_isi (seconds), all_epoch_isis (1D)
    """
    all_spikes = np.asarray(all_spikes, float)
    epochs = np.asarray(epochs, float)

    per_epoch = []
    all_isis = []

    for start, end in epochs:
        i0 = np.searchsorted(all_spikes, start, side="left")
        i1 = np.searchsorted(all_spikes, end,   side="right")
        t = all_spikes[i0:i1]

        isi = np.diff(t)  # seconds, only within this epoch
        all_isis.append(isi)
        per_epoch.append(np.nan if isi.size == 0 else isi.mean())

    all_isis = np.concatenate(all_isis) if len(all_isis) else np.array([])
    overall = np.nan if all_isis.size == 0 else all_isis.mean()
    return np.asarray(per_epoch), overall, all_isis

# Example:
# NREM_epochs could be (n,2) array OR nel EpochArray -> use .data
per_epoch_mean_isi, overall_mean_isi, all_nrem_isis = mean_isi_in_epochs(
    all_spikes,
    nrem_epochs.data if hasattr(nrem_epochs, "data") else nrem_epochs
)

print("Overall mean ISI in NREM (s):", overall_mean_isi)
print("Overall mean ISI in NREM (ms):", overall_mean_isi * 1000)

Overall mean ISI in NREM (s): 0.004694933165618451
Overall mean ISI in NREM (ms): 4.694933165618451


In [334]:
import os, re, numpy as np, pandas as pd
import glob  # <-- add this
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved_error_bar_50ms.mat", f"*_{region}_D2_time_resolved_error_bar_50ms.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Test the complete loading function
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day13_CA1_D2_time_resolved_error_bar_50ms.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.05, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 158, 'total_spikes': 6976785}
   State 0: NREM - requested points=3966, actual used=3966
     Time range: 814.0 to 23078.0
     D2 range: 0.0483 to 0.2280
     D2_ERR range: 0.0140 to 0.0352
   State 1: REM - requested points=414, actual used=414
     Time range: 3254.0 to 20113.0
     D2 range: 0.0178 to 0.1498
     D2_ERR range: 0.0080 to 0.0337
   State 2: WAKE - requested points=4590, actual used=4590
     Time range: 21.0 to 24536.0
     D2 range: 0.0117 to 0.2636
     D2_ERR range: 0.0088 to 0.0449

✅ Complete timeline created:
📊 Total data points: 8970
📊 Time range: 21.0 to 24536.0
📊 Duration: 24515.0 seconds
📊 State distribution:
state
WAKE    4590
NREM    3966
REM      414
Name: count, dtype: int64



In [335]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # --------------------------
# # helper: centers ∈ any interval
# # --------------------------
# def centers_in_intervals(centers, intervals):
#     """
#     centers: (N,) float seconds
#     intervals: (M,2) float seconds, sorted by start (can overlap; ok)
#     returns: (N,) bool
#     """
#     centers = np.asarray(centers, float)
#     iv = np.asarray(intervals, float)
#     if iv.size == 0:
#         return np.zeros_like(centers, dtype=bool)

#     # sort by start just in case
#     order = np.argsort(iv[:, 0])
#     iv = iv[order]
#     starts = iv[:, 0]
#     stops  = iv[:, 1]

#     # for each center, find rightmost start <= center
#     j = np.searchsorted(starts, centers, side="right") - 1
#     ok = (j >= 0) & (centers < stops[np.clip(j, 0, len(stops)-1)])
#     return ok

# # --------------------------
# # 1) Build merged sleep blocks
# # --------------------------
# def merged_sleep_blocks(epoch_df, merge_gap_sec=1e-3):
#     ep = epoch_df[["name","startTime","stopTime","environment"]].copy()
#     ep["startTime"] = pd.to_numeric(ep["startTime"], errors="coerce")
#     ep["stopTime"]  = pd.to_numeric(ep["stopTime"],  errors="coerce")
#     ep = ep.dropna(subset=["startTime","stopTime","environment"]).sort_values("startTime").reset_index(drop=True)

#     blocks = []
#     cur = None

#     for _, r in ep.iterrows():
#         env = r["environment"]
#         st, sp = float(r["startTime"]), float(r["stopTime"])

#         if env != "sleep":
#             if cur is not None:
#                 blocks.append(cur)
#                 cur = None
#             continue

#         if cur is None:
#             cur = {"start": st, "stop": sp, "names":[str(r["name"])]}
#         else:
#             if st <= cur["stop"] + merge_gap_sec:
#                 cur["stop"] = max(cur["stop"], sp)
#                 cur["names"].append(str(r["name"]))
#             else:
#                 blocks.append(cur)
#                 cur = {"start": st, "stop": sp, "names":[str(r["name"])]}

#     if cur is not None:
#         blocks.append(cur)

#     return blocks

# # --------------------------
# # 2) Label blocks as pre/mid/post
# # --------------------------
# def label_sleep_blocks(blocks):
#     if len(blocks) == 0:
#         return []
#     if len(blocks) == 1:
#         blocks[0]["label"] = "pre_sleep"
#         return blocks

#     blocks[0]["label"]  = "pre_sleep"
#     blocks[-1]["label"] = "post_sleep"
#     mids = blocks[1:-1]
#     for i, b in enumerate(mids, start=1):
#         b["label"] = f"mid_sleep_{i}"

#     return blocks

# # --------------------------
# # 3) Assign each D² center to a sleep label + restrict to NREM intervals
# # --------------------------
# def assign_sleep_epoch_to_centers(ca1_complete_df, blocks_labeled, nrem_intervals=None):
#     centers = pd.to_numeric(ca1_complete_df["center_s"], errors="coerce").to_numpy(float)
#     labels = np.full(len(centers), np.nan, dtype=object)

#     for b in blocks_labeled:
#         m = (centers >= b["start"]) & (centers < b["stop"])
#         labels[m] = b["label"]

#     dfp = ca1_complete_df.copy()
#     dfp["sleep_epoch"] = labels
#     dfp = dfp.dropna(subset=["d2", "sleep_epoch"])

#     # ---- NEW: keep only centers inside NREM epochs ----
#     if nrem_intervals is not None:
#         m_nrem = centers_in_intervals(dfp["center_s"].to_numpy(float), nrem_intervals)
#         dfp = dfp.loc[m_nrem].copy()

#     return dfp

# # --------------------------
# # RUN
# # --------------------------
# blocks = merged_sleep_blocks(epoch_df, merge_gap_sec=1e-3)
# blocks = label_sleep_blocks(blocks)

# print("Merged sleep blocks:")
# for b in blocks:
#     print(b["label"], "start", b["start"], "stop", b["stop"], "n_segments", len(b["names"]))

# # pass nrem_epochs.data here
# dfp = assign_sleep_epoch_to_centers(ca1_complete_df, blocks, nrem_intervals=nrem_epochs.data)
# # order dynamically from the labeled blocks you created earlier
# order = [b["label"] for b in blocks]   # e.g. ["pre_sleep","mid_sleep_1","post_sleep"]

# print("\nCounts per sleep epoch (NREM only):")
# print(dfp["sleep_epoch"].value_counts().reindex(order).fillna(0).astype(int))

# # --------------------------
# # Boxplot
# # --------------------------
# data = []
# labels = []
# for lab in order:
#     vals = pd.to_numeric(
#         dfp.loc[dfp["sleep_epoch"] == lab, "d2"],
#         errors="coerce"
#     ).dropna().to_numpy()

#     if vals.size == 0:
#         continue  # skip empty groups so boxplot doesn't error
#     data.append(vals)
#     labels.append(lab)

# fig, ax = plt.subplots(figsize=(max(4.8, 1.2*len(labels)), 4.0))
# ax.boxplot(data, labels=labels, showfliers=False)

# ax.set_xlabel("Sleep epoch (NREM only)")
# ax.set_ylabel("D²")
# ax.set_title("D² across sleep blocks")

# ax.grid(False)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# plt.tight_layout()
# plt.show()



In [336]:
# from pathlib import Path
# import pandas as pd, numpy as np
# from pathlib import Path
# import numpy as np, pandas as pd
# from scipy.io import loadmat

# def load_behavior_trials(basepath, basename="day7"):
#     p = Path(basepath) / f"{basename}.animal.behavior.mat"
#     m = loadmat(str(p), squeeze_me=True, struct_as_record=False)  # classic .mat
#     beh = m.get("behavior", None)
#     if beh is None:
#         raise KeyError("No 'behavior' in MAT file")
#     tr = getattr(beh, "trials", None)
#     speed = getattr(beh, "speed_smooth", None)


#     return speed,tr
# speed,trials = load_behavior_trials(basepath, basename=basename)



In [337]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)  # <-- key fix
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]


In [338]:
# nrem_epochs = nel.EpochArray(state_dict["NREMstate"])
# awake_epochs = nel.EpochArray(state_dict["WAKEstate"])
# # WAKE across the whole session
# # awake_all = nel.EpochArray(state_dict["WAKEstate"])

# # # Build an EpochArray from epoch_df rows 1 and 3
# # sel_idx = [1, 3]  # freeT1 and freeT2 in your screenshot
# # task_iv = epoch_df.loc[sel_idx, ['startTime','stopTime']].to_numpy()
# # task_epochs = nel.EpochArray(task_iv)

# # # Keep only the WAKE that falls inside those two epochs
# # awake_in_task = (awake_all & task_epochs)
# # awake_in_task = awake_in_task[awake_in_task.durations > 0]  # drop empty


# power_spectrum_results = state_power_spectra(
#     E_spikes,  # list/array of per-neuron spike arrays is fine now
#     {"NREMstate": nrem_epochs, "WAKEstate": awake_epochs, "REMstate": rem_epochs},
#     bin_size=0.04, fmin_target=0.1
# )


In [339]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Compute MUA firing rate with 0.04 second bins
time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

Computing MUA firing rate from spike times...


In [340]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# spike_times  # used earlier to make MUA; not needed again here
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges → make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average of MUA (Hz)
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window; allow some tolerance near edges
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

# 4) Interpolate the smoothed MUA onto your D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)
mua_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe (Hz)
ca1_complete_df["MUA"] = mua_at_centers
ca1_complete_df["n_cell"] = n_cell


SWR and barrage

In [341]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs

try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    # Automatically detect the HSE filename based on basename
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    # brpath=os.path.join(basepath, "BARR_archive","Barrage_Files")
    # mat_path = os.path.join(brpath, f'{basename}.HSEn2.events.mat')

    # data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    # barrage = data['HSEn2']
    # barrage_timestamps= barrage.timestamps 
    # barrage_epochs=nel.EpochArray(barrage_timestamps)
    # barrage_epochs=nrem_epochs&barrage_epochs
    # Create barrage epochs
    # Find barrages in NREM epochs
    # #    ---- 1) Define paths ----
    # barrage_path = os.path.join(basepath, "Barrage_Files")

    # # Automatically detect the HSE filename based on basename
    # basename = os.path.basename(basepath)
    # matfile = f"{basename}.clean.HSE.mat"
    # fullfile = os.path.join(barrage_path, matfile)

    # # # ---- 2) Load .mat file ----

    # # # MATLAB variables available:
    # # #   timestamps : Nx2 array
    # mat = sio.loadmat(fullfile, squeeze_me=True, struct_as_record=False)
    # HSE = mat["HSE"]

    # timestamps = np.asarray(HSE.timestamps)
    # keep       = np.asarray(HSE.keep).squeeze().astype(int)   # indices into timestamps (1-based)
    # NREM       = np.asarray(HSE.NREM).squeeze().astype(int)   # indices into keep (1-based)

    # # # --- MATLAB: keep(NREM) ---
    # # # In MATLAB: idx_ts_1 = keep(NREM)
    # # # Step 1: convert NREM to 0-based to index keep
    # idx_keep_0 = NREM - 1              # 0-based indices into keep
    # idx_ts_1   = keep[idx_keep_0]      # still 1-based indices into timestamps

    # # # Step 2: convert to 0-based to index timestamps in Python
    # idx_ts_0   = idx_ts_1 - 1

    # # # --- MATLAB: timestamps(keep(NREM), :) ---
    # nrem_intervals = timestamps[idx_ts_0, :]

    # # # ---- Convert to nel.EpochArray ----
    # barrage_epoch = nel.EpochArray(nrem_intervals)
    barrage_start = barrages_in_nrem.data[:, 0]
        
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times


No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for U:\data\HMC\HMC1\day13


novel and familiar memory

In [379]:
# simplest: replace the substring anywhere in the name
# epoch_df["name"] = epoch_df["name"].str.replace("midseelp", "midsleep", regex=False)


In [380]:
epoch_df

,name,startTime,stopTime,environment,behavioralParadigm,manipulation,stimuli,notes,basepath
0,preSleep_210416_073406,0.0000,10570.13115,sleep,NaN,NaN,NaN,NaN,U:\data\HMC\HMC1\day13
1,maze_210416_130639,10570.1312,13029.35035,linear,6,NaN,NaN,NaN,U:\data\HMC\HMC1\day13
2,postSleep_210416_135513,13029.3504,24558.07355,sleep,NaN,NaN,NaN,NaN,U:\data\HMC\HMC1\day13


In [381]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# import re

# def label_epoch_from_name(name: str) -> str:
#     s = (str(name) or "").lower()

#     if "presleep" in s:  return "presleep"
#     if "midsleep"  in s: return "midsleep"
#     if "postsleep" in s: return "postsleep"

#     # --- task / T-maze markers ---
#     # match 't_' or 't1_' / 't2_' / ... anywhere (word-like), plus common aliases
#     if (re.search(r'(?:^|[^a-z])t(?:_|[0-9])', s)          # t_ or t#_
#         or "tmaze" in s or "t-maze" in s
#         or "freet" in s or "force" in s
#         or "task" in s):
#         return "task"

#     return "other"

# # --- normalize possible Data Wrangler column names ---
# def _normalize_epoch_columns(df: pd.DataFrame) -> pd.DataFrame:
#     tmp = df.copy()
#     # strip spaces and leading '#' from column names
#     tmp.columns = [c.strip().lstrip("#").strip() for c in tmp.columns]
#     # make sure required columns exist (startTime/stopTime/name)
#     rename_map = {}
#     for c in tmp.columns:
#         cl = c.lower()
#         if cl == "starttime": rename_map[c] = "startTime"
#         if cl == "stoptime":  rename_map[c] = "stopTime"
#         if cl == "name":      rename_map[c] = "name"
#     if rename_map:
#         tmp = tmp.rename(columns=rename_map)
#     missing = [c for c in ("name", "startTime", "stopTime") if c not in tmp.columns]
#     if missing:
#         raise KeyError(f"epoch_df missing required columns: {missing}")
#     return tmp

# def annotate_epochs(ca_df: pd.DataFrame, epoch_df: pd.DataFrame,
#                     enumerate_labels=("postsleeppp",),  # labels to number by order
#                     max_tasks=3):
#     tmp = _normalize_epoch_columns(epoch_df)[["name","startTime","stopTime"]].copy()
#     tmp["epoch_group"] = [label_epoch_from_name(n) for n in tmp["name"]]
#     tmp = tmp.dropna(subset=["startTime","stopTime"])
#     tmp = tmp[tmp["startTime"] < tmp["stopTime"]].copy()
#     tmp = tmp.sort_values("startTime").reset_index(drop=True)

#     # number selected labels (e.g., midsleep1, midsleep2, ...)
#     for lab in enumerate_labels:
#         m = tmp["epoch_group"].eq(lab)
#         if m.any():
#             idxs = tmp.index[m].tolist()
#             for k, i in enumerate(idxs, start=1):
#                 tmp.loc[i, "epoch_group"] = f"{lab}{k}"

#     # number first few task blocks as before
#     m_task = tmp["epoch_group"].eq("task")
#     if m_task.any():
#         task_order = tmp.index[m_task].tolist()
#         for i, idx in enumerate(task_order[:max_tasks], start=1):
#             tmp.loc[idx, "epoch_group"] = f"task{i}"

#     intervals = pd.IntervalIndex.from_arrays(tmp["startTime"].to_numpy(),
#                                              tmp["stopTime"].to_numpy(),
#                                              closed="left")
#     tmp = tmp.set_index(intervals)

#     ca = ca_df.copy()
#     centers = ca["center_s"].to_numpy()
#     idx = tmp.index.get_indexer(centers)
#     ca["epoch_group"] = np.where(idx >= 0, tmp.iloc[idx]["epoch_group"].to_numpy(), np.nan)
#     return ca


# # -------- 1) annotate ca1_complete_df with epoch_group --------
# ca_annot = annotate_epochs(ca1_complete_df, epoch_df)

# # -------- 2) filter to groups & make boxplots --------
# wanted_groups = ["presleep", "midsleep", "postsleep"]  # add "task" if you want a 4th box

# def plot_box_for_state(df_annot, state_label, groups=wanted_groups, title=None):
#     d = (df_annot
#          .loc[(df_annot["state"].astype(str).str.upper()==state_label.upper())
#               & (df_annot["epoch_group"].isin(groups))]
#          .copy())
#     if d.empty:
#         print(f"No rows for state={state_label}")
#         return

#     # order & tidy
#     d["epoch_group"] = pd.Categorical(d["epoch_group"], categories=groups, ordered=True)

#     # --- boxplot (matplotlib) ---
#     fig, ax = plt.subplots(figsize=(5.2,4.2))
#     data = [d.loc[d["epoch_group"]==g, "d2"].dropna().values for g in groups]
#     ax.boxplot(data, labels=[g.replace("sleep"," sleep") for g in groups],
#                showfliers=False, widths=0.6)
#     ax.set_xlabel("Epoch")
#     ax.set_ylabel("D²")
#     ax.set_title(title or f"{state_label}: D² by epoch")
#     for s in ("top","right"): ax.spines[s].set_visible(False)
#     plt.tight_layout()
#     plt.show()

#     # optional: quick stats readout (Kruskal–Wallis + pairwise Wilcoxon, paired only if the same file/session needed)
#     try:
#         from scipy.stats import kruskal, mannwhitneyu
#         stats_groups = [g for g in data if len(g) >= 5]
#         if len(stats_groups) >= 2:
#             H, p = kruskal(*stats_groups)
#             print(f"[{state_label}] Kruskal–Wallis across groups: H={H:.2f}, p={p:.3e}")
#         # pairwise (unpaired) Mann–Whitney as a lightweight summary
#         for i in range(len(groups)):
#             for j in range(i+1, len(groups)):
#                 gi, gj = data[i], data[j]
#                 if len(gi)>=5 and len(gj)>=5:
#                     U, p = mannwhitneyu(gi, gj, alternative="two-sided")
#                     print(f"  {groups[i]} vs {groups[j]}: U={U:.0f}, p={p:.3e}")
#     except Exception as e:
#         print("Stats skipped:", e)

# # NREM: compare pre/mid/post sleep
# ca_annot = annotate_epochs(ca1_complete_df, epoch_df)
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ---------- build normalized df ----------
# def first_n_per_epoch(df, n=1200):
#     """Return first n rows per epoch_group after sorting by center_s."""
#     return (df.sort_values("center_s")
#               .groupby("epoch_group", group_keys=False, observed=True)
#               .head(n))

# sleep_groups = ["presleep", "midsleep", "postsleep"]

# # 1) NREM only, sleep epochs only
# d = ca_annot.copy()
# d["state"] = d["state"].astype(str).str.upper()
# d = d[(d["state"] == "NREM") & (d["epoch_group"].isin(sleep_groups))].copy()

# # 2) take first 1200 rows per epoch
# d = first_n_per_epoch(d, n=1200)

# # 3) presleep mean
# pre = d[d["epoch_group"] == "presleep"]["d2"].dropna().to_numpy()
# if pre.size == 0:
#     raise ValueError("No presleep NREM rows found (first 1200).")
# m_pre = float(np.nanmean(pre))

# # safety for zero (very unlikely, but guard anyway)
# eps = 1e-12 if m_pre == 0 else 0.0

# # 4) normalize midsleep & postsleep
# d2_rel = d[d["epoch_group"].isin(["midsleep", "postsleep"])].copy()
# d2_rel["d2_rel"] = (d2_rel["d2"] - m_pre) / (d2_rel["d2"] + m_pre + eps)
# d2_rel["epoch_group"] = pd.Categorical(
#     d2_rel["epoch_group"], categories=["midsleep", "postsleep"], ordered=True
# )

# # ---------- plot (only midsleep & postsleep) ----------
# plt.figure(figsize=(5.4, 4.0))
# sns.boxplot(data=d2_rel, x="epoch_group", y="d2_rel", showfliers=False, width=0.6)
# sns.stripplot(data=d2_rel, x="epoch_group", y="d2_rel",
#               color="k", alpha=0.25, size=2, linewidth=0, dodge=False)
# plt.axhline(0, lw=1, ls="--", color="gray")  # zero = equal to presleep mean
# plt.xlabel("")
# plt.ylabel(r"(sleep − presleep) / (sleep + presleep)")
# plt.title("NREM (first 1200 rows): midsleep & postsleep vs presleep baseline")
# ax = plt.gca()
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
# plt.tight_layout(); plt.show()

# # (optional) quick text readout
# print(f"Presleep mean D² (NREM, first 1200 rows): {m_pre:.4g}")
# for g in ["midsleep", "postsleep"]:
#     vals = d2_rel.loc[d2_rel["epoch_group"]==g, "d2_rel"].dropna().to_numpy()
#     if vals.size:
#         print(f"{g}: n={vals.size}, median={np.median(vals):.3f}, mean={np.mean(vals):.3f}")


In [382]:
# from pathlib import Path
# out_dir=r"D:\Documents\GitHub\Sleep_criticaltiy\data\novel_familiar\novel"
# # --- save d2_rel ---
# Path(out_dir).mkdir(parents=True, exist_ok=True)
# save_path = Path(out_dir) / f"{animal_name}_{day}.pkl"
# d2_rel.to_pickle(save_path)
# print(f"Saved: {save_path}")


In [383]:
# import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# # --- trials -> IntervalIndex (handles ragged shapes) ---
# def trials_interval_index(trials, closed="left"):
#     a = np.asarray(trials, float)
#     a = a.reshape(-1, 2)                      # (N,2)
#     ok = np.isfinite(a).all(1) & (a[:,1] > a[:,0])
#     a = a[ok]
#     return pd.IntervalIndex.from_arrays(a[:,0], a[:,1], closed=closed)

# import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# # --- helpers ---
# def trials_interval_index(trials, closed="left"):
#     a = np.asarray(trials, float).reshape(-1, 2)
#     ok = np.isfinite(a).all(1) & (a[:,1] > a[:,0])
#     a = a[ok]
#     return pd.IntervalIndex.from_arrays(a[:,0], a[:,1], closed=closed)

# def epoch_bounds(epoch_df):
#     """Return dict: {'task1': (start, stop), 'task2': ..., ...}"""
#     tmp = epoch_df.copy()
#     cols = {c.lower(): c for c in tmp.columns}
#     sC, eC, nC = cols['starttime'], cols['stoptime'], cols['name']
#     # label + number tasks in time order
#     lab = tmp[nC].astype(str).str.lower()
#     is_task = (lab.str.contains(r'(?:^|[^a-z])t(?:_|[0-9])')) | lab.str.contains('tmaze|t-maze|task|freet|forcet')
#     order = tmp[is_task].sort_values(sC).index.tolist()
#     out = {}
#     for i, idx in enumerate(order[:3], start=1):
#         out[f"task{i}"] = (float(tmp.at[idx, sC]), float(tmp.at[idx, eC]))
#     return out

# def select_first10_in_and_out(d_epoch, trial_iv_epoch, n_out=5):
#     """Rows inside first-10 trials + first-10 out-of-trial samples by time."""
#     # in-trial membership
#     loc = trial_iv_epoch.get_indexer(d_epoch["center_s"].to_numpy(float))
#     in_mask = (loc >= 0)

#     # keep *all* rows that fall in those selected trials
#     d_in  = d_epoch.loc[in_mask].copy()

#     # among out-of-trial rows, keep the *earliest* n_out by time
#     d_out = d_epoch.loc[~in_mask].sort_values("center_s").head(n_out).copy()
#     d_in["trial_bin"]  = "in-trial"
#     d_out["trial_bin"] = "out-of-trial"
#     return pd.concat([d_in, d_out], ignore_index=True)

# # --- build objects ---
# trial_iv_all = trials_interval_index(trials)
# epoch_iv = epoch_bounds(epoch_df)  # needs epoch_df with startTime/stopTime/name

# # base WAKE dataframe limited to task epochs we care about
# wake_groups = ["task1","task2","task3"]
# d0 = ca_annot[(ca_annot["state"].astype(str).str.upper()=="WAKE") &
#               (ca_annot["epoch_group"].isin(wake_groups))].copy()

# parts = []
# for g in wake_groups:
#     if g not in epoch_iv:  # skip if this task doesn't exist
#         continue
#     s, e = epoch_iv[g]

#     # rows of this epoch
#     de = d0[d0["epoch_group"].eq(g)].copy()
#     if de.empty:
#         continue

#     # trials that start within this epoch → first 10
#     ta = np.asarray(trials, float).reshape(-1, 2)
#     ok = np.isfinite(ta).all(1) & (ta[:,1] > ta[:,0])
#     ta = ta[ok]
#     t_in_epoch = ta[(ta[:,0] >= s) & (ta[:,0] < e)]
#     t_in_epoch = t_in_epoch[:10] if len(t_in_epoch) > 10 else t_in_epoch
#     if t_in_epoch.size == 0:
#         # no trials found → just take the first 10 out-of-trial samples
#         parts.append(de.sort_values("center_s").head(10).assign(trial_bin="out-of-trial"))
#         continue

#     trial_iv_epoch = pd.IntervalIndex.from_arrays(t_in_epoch[:,0], t_in_epoch[:,1], closed="left")

#     parts.append(select_first10_in_and_out(de, trial_iv_epoch, n_out=10))

# d = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
# if d.empty:
#     print("No data after selection.")
# else:
#     d["epoch_group"] = pd.Categorical(d["epoch_group"], categories=wake_groups, ordered=True)
#     d["trial_bin"]   = pd.Categorical(d["trial_bin"],   categories=["in-trial","out-of-trial"], ordered=True)

#     # --- plot ---
#     plt.figure(figsize=(6.2,4.2))
#     sns.boxplot(data=d, x="epoch_group", y="d2", hue="trial_bin",
#                 order=wake_groups, hue_order=["in-trial","out-of-trial"],
#                 showfliers=False)
#     sns.stripplot(data=d, x="epoch_group", y="d2", hue="trial_bin",
#                   order=wake_groups, hue_order=["in-trial","out-of-trial"],
#                   dodge=True, color="k", alpha=0.2, size=2, linewidth=0)
#     plt.xlabel("Task epoch")
#     plt.ylabel("D²")
#     plt.title("WAKE: D² (first 10 trials) vs first 10 out-of-trial samples")
#     plt.legend(title="", frameon=False)
#     ax = plt.gca(); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); plt.show()


In [384]:
# import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# # ===================== PARAMS =====================
# running_thresh = 2.5      # speed threshold (units of your speed)
# smooth_window_s = 0.5      # smoothing window (s) for speed
# min_bout_s = 1           # remove running/quiescent bouts shorter than this
# wake_only = True            # if ca_annot has 'state', limit to WAKE
# use_first10_trials_per_task = True
# wake_groups = ["task1","task2","task3"]
# # =================================================

# # ---------- helpers ----------
# def trials_interval_index(trials, closed="left"):
#     a = np.asarray(trials, float).reshape(-1, 2)
#     ok = np.isfinite(a).all(1) & (a[:,1] > a[:,0])
#     a = a[ok]
#     return pd.IntervalIndex.from_arrays(a[:,0], a[:,1], closed=closed)

# def epoch_bounds(epoch_df):
#     """Return dict: {'task1': (start, stop), ...} by start time order."""
#     tmp = epoch_df.copy()
#     cols = {c.lower(): c for c in tmp.columns}
#     sC, eC, nC = cols['starttime'], cols['stoptime'], cols['name']
#     lab = tmp[nC].astype(str).str.lower()
#     is_task = (lab.str.contains(r'(?:^|[^a-z])t(?:_|[0-9])')) | lab.str.contains('tmaze|t-maze|task|freet|forcet')
#     order = tmp[is_task].sort_values(sC).index.tolist()
#     out = {}
#     for i, idx in enumerate(order[:3], start=1):
#         out[f"task{i}"] = (float(tmp.at[idx, sC]), float(tmp.at[idx, eC]))
#     return out

# def moving_average_uniform(x, win):
#     if win <= 1: return x.copy()
#     w = np.ones(int(win), float) / int(win)
#     return np.convolve(x, w, mode="same")

# def enforce_min_bout(mask_bool, t, min_len_s):
#     m = mask_bool.astype(bool).copy()
#     idx = np.flatnonzero(np.r_[True, m[1:] != m[:-1], True])
#     for i in range(len(idx)-1):
#         s, e = idx[i], idx[i+1]
#         dur = t[e-1] - t[s] if e > s else 0.0
#         if dur < min_len_s:
#             val = m[s-1] if s > 0 else (m[e] if e < m.size else m[s])
#             m[s:e] = val
#     return m

# def running_label_at_times(speed, center_times, *, thr, smooth_s, min_bout_s):
#     """Return array of 'running'/'quiescence' at each center time."""
#     t = np.asarray(speed[:,0], float)
#     v = np.asarray(speed[:,1], float)
#     order = np.argsort(t)
#     t = t[order]; v = v[order]
#     dt = np.median(np.diff(t[np.isfinite(t)]))
#     win = max(1, int(round(smooth_s / dt)))
#     v_sm = moving_average_uniform(v, win)
#     run_mask = enforce_min_bout(v_sm >= thr, t, min_bout_s)

#     # sample the boolean mask at each center time by “last known value” (left-closed)
#     idx = np.searchsorted(t, np.asarray(center_times, float), side="right") - 1
#     idx = np.clip(idx, 0, len(t)-1)
#     return np.where(run_mask[idx], "running", "quiescence")

# # ---------- build selection (first-10 trials in each task; keep same as your code) ----------
# trial_iv_all = trials_interval_index(trials)
# epoch_iv = epoch_bounds(epoch_df)

# d0 = ca_annot.copy()
# if 'state' in d0.columns and wake_only:
#     d0 = d0[d0['state'].astype(str).str.upper() == 'WAKE'].copy()
# if 'epoch_group' in d0.columns:
#     d0 = d0[d0['epoch_group'].isin(wake_groups)].copy()

# if use_first10_trials_per_task and 'epoch_group' in d0.columns:
#     parts = []
#     ta_all = np.asarray(trials, float).reshape(-1, 2)
#     ok = np.isfinite(ta_all).all(1) & (ta_all[:,1] > ta_all[:,0])
#     ta_all = ta_all[ok]

#     for g in wake_groups:
#         if g not in epoch_iv: 
#             continue
#         s, e = epoch_iv[g]
#         de = d0[d0['epoch_group'].eq(g)].copy()
#         if de.empty: 
#             continue
#         t_in_epoch = ta_all[(ta_all[:,0] >= s) & (ta_all[:,0] < e)]
#         t_in_epoch = t_in_epoch[:10] if len(t_in_epoch) > 10 else t_in_epoch

#         if t_in_epoch.size == 0:
#             parts.append(de.sort_values('center_s').head(10).assign(in_out='out-of-trial'))
#             continue

#         trial_iv_epoch = pd.IntervalIndex.from_arrays(t_in_epoch[:,0], t_in_epoch[:,1], closed="left")
#         loc = trial_iv_epoch.get_indexer(de['center_s'].to_numpy(float))
#         de['in_out'] = np.where(loc >= 0, 'in-trial', 'out-of-trial')
#         # keep all in-trial rows + earliest 10 out-of-trial
#         de_in  = de[de['in_out'] == 'in-trial'].copy()
#         de_out = de[de['in_out'] == 'out-of-trial'].sort_values('center_s').head(10).copy()
#         parts.append(pd.concat([de_in, de_out], ignore_index=True))

#     d = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
# else:
#     # simpler path: use all rows and mark in/out from *all* trials
#     d = d0.copy()
#     if not d.empty:
#         loc = trial_iv_all.get_indexer(d['center_s'].to_numpy(float))
#         d['in_out'] = np.where(loc >= 0, 'in-trial', 'out-of-trial')

# if d.empty:
#     print("No data after selection.")
# else:
#     # ---------- add running/quiescence at each center_s ----------
#     d['run_bin'] = running_label_at_times(
#         speed, d['center_s'].to_numpy(float),
#         thr=running_thresh, smooth_s=smooth_window_s, min_bout_s=min_bout_s
#     )

#     # tidy categories
#     d['in_out'] = pd.Categorical(d['in_out'], categories=['in-trial','out-of-trial'], ordered=True)
#     d['run_bin'] = pd.Categorical(d['run_bin'], categories=['quiescence','running'], ordered=True)

#     # ---------- PLOTS ----------
#     # A) Box/strip of D² by in/out with hue=running
#     plt.figure(figsize=(6.8, 4.2))
#     sns.boxplot(data=d, x='in_out', y='d2', hue='run_bin', showfliers=False)
#     sns.stripplot(data=d, x='in_out', y='d2', hue='run_bin',
#                   dodge=True, color='k', alpha=0.25, size=2, linewidth=0)
#     plt.xlabel("")
#     plt.ylabel("D²")
#     plt.title("D² at center_s: in-trial vs out-of-trial × running state")
#     # de-duplicate legends from double plotting
#     handles, labels = plt.gca().get_legend_handles_labels()
#     bylab = dict(zip(labels, handles))
#     plt.legend(bylab.values(), bylab.keys(), frameon=False, title="")
#     ax = plt.gca(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
#     plt.tight_layout(); plt.show()

#     # (optional) If you also want per-task panels when epoch_group exists:
#     if 'epoch_group' in d.columns:
#         d['epoch_group'] = pd.Categorical(d['epoch_group'], categories=wake_groups, ordered=True)
#         g = sns.catplot(
#             data=d, x='in_out', y='d2', hue='run_bin',
#             col='epoch_group', kind='box', sharey=True, showfliers=False,
#             height=3.2, aspect=0.9
#         )
#         g.set_axis_labels("", "D²")
#         g.fig.set_size_inches(12, 6)   # total W×H for the whole figure
#         g.fig.subplots_adjust(top=0.82)
#         g.fig.suptitle("D² by in/out × running state (per task)")
#         for ax in g.axes.flatten():
#             ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
#         plt.show()


ISI entropy

In [385]:
# out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats")
# out_dir.mkdir(parents=True, exist_ok=True)

# fname = f"{animal_name}_{day}_{brain_region}.pkl"
# save_path = out_dir / fname

# ca1_complete_df.to_pickle(save_path)
# print(f"Saved to: {save_path}")

In [386]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from typing import Iterable

# # =========================
# # ISI entropy
# # =========================
# def isi_entropy(spike_times, n_bins=20):
#     """
#     Shannon entropy (bits) of log10(ISI) distribution.
#     Returns NaN if fewer than 2 ISIs (i.e., <3 spikes).
#     """
#     st = np.asarray(spike_times, float)
#     st = st[np.isfinite(st)]
#     st = np.unique(np.sort(st))
#     if st.size < 3:
#         return np.nan
#     isis = np.diff(st)
#     isis = isis[np.isfinite(isis) & (isis > 0)]
#     if isis.size < 2:
#         return np.nan
#     x = np.log10(isis)
#     hist, _ = np.histogram(x, bins=n_bins)
#     N = hist.sum()
#     if N <= 0:
#         return np.nan
#     p = hist.astype(float) / N
#     p = p[p > 0]
#     return float(-np.sum(p * np.log2(p)))  # bits

# # =========================
# # Utilities
# # =========================
# def _flatten_spikes(spike_times):
#     """Pool spike times across inputs and return sorted unique float seconds."""
#     if isinstance(spike_times, dict):
#         parts = [np.asarray(v, float).ravel() for v in spike_times.values()]
#         x = np.concatenate([p[np.isfinite(p)] for p in parts]) if parts else np.array([], float)
#     else:
#         arr = np.asarray(spike_times, dtype=object)
#         if arr.dtype == object:
#             parts = []
#             for it in arr:
#                 if it is None:
#                     continue
#                 v = np.asarray(it, float).ravel()
#                 v = v[np.isfinite(v)]
#                 if v.size:
#                     parts.append(v)
#             if parts:
#                 x = np.concatenate(parts)
#             else:
#                 try:
#                     x = np.asarray(spike_times, float).ravel()
#                 except (TypeError, ValueError):
#                     x = np.array([], float)
#         else:
#             x = arr.astype(float).ravel()
#     x = x[np.isfinite(x)]
#     return np.sort(np.unique(x))

# def _coerce_epochs(ep):
#     """Coerce many epoch formats to a sorted list[(start, stop)]. Accepts ep, or ep.data with ≥2 columns."""
#     # Try .data first (Epoch-like objects)
#     if hasattr(ep, "data"):
#         vals = getattr(ep, "data")
#         try:
#             if isinstance(vals, pd.DataFrame) and vals.shape[1] >= 2:
#                 a = vals.iloc[:, 0].astype(float).to_numpy()
#                 b = vals.iloc[:, 1].astype(float).to_numpy()
#                 out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#                 out.sort(key=lambda t: t[0]); return out
#             vals = np.asarray(vals)
#             if vals.ndim == 2 and vals.shape[1] >= 2:
#                 a = vals[:, 0].astype(float); b = vals[:, 1].astype(float)
#                 out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#                 out.sort(key=lambda t: t[0]); return out
#         except Exception:
#             pass

#     # DataFrame
#     if isinstance(ep, pd.DataFrame):
#         if {'start','stop'}.issubset(ep.columns):
#             a = ep['start'].astype(float).to_numpy()
#             b = ep['stop'].astype(float).to_numpy()
#         elif {'startTime','stopTime'}.issubset(ep.columns):
#             a = ep['startTime'].astype(float).to_numpy()
#             b = ep['stopTime'].astype(float).to_numpy()
#         else:
#             a = ep.iloc[:,0].astype(float).to_numpy()
#             b = ep.iloc[:,1].astype(float).to_numpy()
#         out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#         out.sort(key=lambda t: t[0]); return out

#     # Iterable of pairs
#     if isinstance(ep, Iterable):
#         out = []
#         for item in ep:
#             try:
#                 s, e = item
#                 s, e = float(s), float(e)
#                 if np.isfinite(s) and np.isfinite(e) and e > s:
#                     out.append((s, e))
#             except Exception:
#                 pass
#         out.sort(key=lambda t: t[0]); return out

#     return []

# def _count_events(event_epochs, a, b):
#     """Count events whose mid-point falls in [a,b)."""
#     c = 0
#     for s, e in event_epochs:
#         if not np.isfinite(s) or not np.isfinite(e) or e <= s:
#             continue
#         mid = 0.5 * (s + e)
#         if a <= mid < b:
#             c += 1
#     return c
# def _centers_within_long_epochs(centers_s, epochs_data, min_dur=60.0):
#     """
#     centers_s: 1D array of center times (sec)
#     epochs_data: (N,2) array-like [[start,end], ...] in seconds
#     """
#     ep = np.asarray(epochs_data, dtype=float)
#     if ep.size == 0:
#         return np.zeros(len(centers_s), dtype=bool)

#     # keep only epochs with duration >= min_dur
#     dur = ep[:, 1] - ep[:, 0]
#     ep = ep[dur >= float(min_dur)]
#     if ep.size == 0:
#         return np.zeros(len(centers_s), dtype=bool)

#     # assume epochs are sorted; if not, sort by start
#     order = np.argsort(ep[:, 0])
#     ep = ep[order]
#     starts, ends = ep[:, 0], ep[:, 1]

#     c = np.asarray(centers_s, dtype=float)

#     # membership via searchsorted (O(M log N))
#     idx = np.searchsorted(starts, c, side="right") - 1
#     ok = (idx >= 0) & (c < ends[np.clip(idx, 0, len(ends)-1)])
#     return ok
# # =========================
# # Core: ISI entropy per window + ripple/barrage rates
# # =========================
# def isi_entropy_windows_with_tiles(
#     spike_times,
#     centers_df,                 # must have columns: center_s, state
#     ripple_epochs,
#     barrage_epochs,
#     state_filter="NREM",
#     window=10.0,                # seconds (±15s)
#     min_spikes=3,               # require ≥3 spikes in window to form ≥2 ISIs
#     n_bins_isi=20               # histogram bins for log10(ISI)
# ):
#     st = _flatten_spikes(spike_times)
#     out_cols = ["center_s","state","isi_entropy","n_spikes","ripple_rate","barrage_rate"]
#     if st.size == 0:
#         return pd.DataFrame(columns=out_cols)

#     # centers (NREM only by default)
#     # centers (NREM only by default)
#     # --- your block, with the extra epoch filter ---
#     # NREM_epochs.data is (n,2) start/end in seconds
#     long_nrem_data = nrem_epochs.data[(nrem_epochs.data[:, 1] - nrem_epochs.data[:, 0]) >= 120]
#     if "state" in centers_df.columns:
#         sel = centers_df["state"].astype(str).str.upper() == str(state_filter).upper()
#         base = centers_df.loc[sel & centers_df["center_s"].notna()].copy()
#     else:
#         base = centers_df.loc[centers_df["center_s"].notna()].copy()
#         base["state"] = state_filter

#     in_long_nrem = _centers_within_long_epochs(base["center_s"].to_numpy(float), long_nrem_data, min_dur=120)
#     cdf = base.loc[in_long_nrem].copy()

#     if centers.size == 0:
#         return pd.DataFrame(columns=out_cols)

#     rip = _coerce_epochs(ripple_epochs)
#     bar = _coerce_epochs(barrage_epochs)

#     half = window/2.0
#     rows = []
#     for i, (c, s) in enumerate(zip(
#         cdf["center_s"].to_numpy(float),
#         cdf["state"].astype(str).to_numpy()
#     )):
#         a, b = c - half, c + half
#         # spikes in window
#         wst = st[(st >= a) & (st < b)]
#         nsp = int(wst.size)

#         if nsp >= min_spikes:
#             H = isi_entropy(wst, n_bins=n_bins_isi)
#         else:
#             H = np.nan

#         dur = b - a
#         if dur > 0:
#             rr = _count_events(rip, a, b) / dur
#             br = _count_events(bar, a, b) / dur
#         else:
#             rr = np.nan; br = np.nan

#         # start with all original metadata from centers_df (current row index)
#         row_dict = cdf.iloc[i].to_dict()

#         # add new fields
#         row_dict.update({
#             "isi_entropy": H,
#             "n_spikes": nsp,
#             "ripple_rate": rr,
#             "barrage_rate": br
#         })

#         rows.append(row_dict)

#     out = pd.DataFrame(rows).sort_values("center_s").reset_index(drop=True)
#     print(out.head())
#     return out

# # =========================
# # Tile aggregation (6 tiles) and plotting
# # =========================

# # =========================
# # Example usage (adapt to your data)
# # =========================
# # Required inputs you already have in your workspace:
# #   - spike_times: array/list/dict of spike timestamps (sec)
# #   - ca1_complete_df: DataFrame with ['center_s','state']
# #   - ripple_epochs, barrage_epochs: epoch objects (with .data), DataFrames, or list[(start,stop)]
# #import numpy as np
# import pandas as pd

# def filter_df_by_epoch_names(df, epoch_df, names=("pre","post"),
#                             tcol="center_s", start_col="startTime", stop_col="stopTime", name_col="name"):
#     out = {}
#     t = df[tcol].to_numpy(float)

#     for nm in names:
#         iv = epoch_df.loc[epoch_df[name_col].eq(nm), [start_col, stop_col]].to_numpy(float)
#         if iv.size == 0:
#             out[nm] = df.iloc[0:0].copy()
#             continue

#         mask = np.zeros(len(df), dtype=bool)
#         for start, stop in iv:
#             mask |= (t >= start) & (t < stop)   # [start, stop)
#         out[nm] = df.loc[mask].copy()

#     return out

# def mask_times_in_intervals(t, intervals):
#     """
#     t: 1D times
#     intervals: (n,2) array [[start, stop], ...]
#     returns: boolean mask, same length as t
#     """
#     t = np.asarray(t, float)
#     iv = np.asarray(intervals, float)
#     if iv.size == 0:
#         return np.zeros_like(t, dtype=bool)

#     # ensure sorted, non-overlapping assumption not required
#     iv = iv[np.argsort(iv[:, 0])]

#     mask = np.zeros(len(t), dtype=bool)
#     for start, stop in iv:
#         mask |= (t >= start) & (t < stop)   # [start, stop)
#     return mask

# # iv = nrem_epochs_with_barrage[nrem_epochs_with_barrage.durations>120].data
# # mask = mask_times_in_intervals(ca1_complete_df["center_s"].to_numpy(float), iv)

# # ca1_nrem_barrage = ca1_complete_df.loc[mask].copy()

# # centers_df = ca1_nrem_barrage
# out_df = isi_entropy_windows_with_tiles(
#     spike_times=spike_times,
#     centers_df=ca1_complete_df,
#     ripple_epochs=ripple_epochs,       # pass the objects; the function will read .data if present
#     barrage_epochs=None,
#     state_filter="NREM",
#     window=30.0,                       # ±15 s
#     min_spikes=3,                      # need ≥3 spikes in window
#     n_bins_isi=40
# )




In [387]:
# import pandas as pd
# import numpy as np

# def make_equal_width_tiles(series, k=6):
#     """
#     Creates exactly k equal-width tiles (B1..Bk) for the given series.
#     The width is determined by (max - min) / k.
#     """
#     # 1. Setup Data and Categories
#     v = pd.Series(series.to_numpy(float))
#     ok = np.isfinite(v)
    
#     # Define the exact categories B1...Bk
#     out_cats = [f"B{i}" for i in range(1, k+1)]
    
#     # Initialize output as Categorical with the known categories (allows empty bins)
#     out = pd.Series(
#         pd.Categorical([np.nan] * len(v), categories=out_cats, ordered=True),
#         index=series.index
#     )

#     if ok.sum() == 0:
#         return out

#     vals = v[ok]
#     min_val, max_val = vals.min(), vals.max()

#     # 2. Case: Constant values (Min == Max)
#     # pd.cut fails when range is 0. We force constant data into the middle bin.
#     if min_val == max_val:
#         mid_idx = int((k + 1) // 2)  # e.g., for k=6, puts data in B3
#         target_label = f"B{mid_idx}"
#         out.loc[vals.index] = target_label
#         return out

#     # 3. Main Logic: Equal Width Binning
#     # pd.cut splits the range (min to max) into k equal linear intervals
#     try:
#         cut_vals = pd.cut(
#             vals, 
#             bins=k, 
#             labels=out_cats, 
#             include_lowest=True
#         )
#         out.loc[vals.index] = cut_vals
#     except Exception as e:
#         print(f"Binning failed: {e}")
#         pass

#     return out

# def aggregate_tiles(df, by="ripple", value_col="isi_entropy", k=6):
#     """
#     Aggregate 'value_col' by k quantile tiles of 'ripple_rate' or 'barrage_rate'.
#     Returns a tidy DataFrame with mean, SEM, and n per tile.
#     """
#     assert by in ("ripple", "barrage")
#     rate_col = "ripple_rate" if by == "ripple" else "barrage_rate"
#     tile_col = "tile_ripple" if by == "ripple" else "tile_barrage"

#     dd = df.dropna(subset=[value_col, rate_col]).copy()
#     if dd.empty:
#         cats = [f"Q{i}" for i in range(1, k+1)]
#         return pd.DataFrame({tile_col: pd.Categorical([], categories=cats, ordered=True),
#                              f"{value_col}_mean": pd.Series([], dtype=float),
#                              f"{value_col}_sem": pd.Series([], dtype=float),
#                              "n": pd.Series([], dtype=int)})

#     dd[tile_col] = make_equal_width_tiles(dd[rate_col], k=k)
#     if dd[tile_col].isna().all():
#         cats = [f"Q{i}" for i in range(1, k+1)]
#         return pd.DataFrame({tile_col: pd.Categorical([], categories=cats, ordered=True),
#                              f"{value_col}_mean": pd.Series([], dtype=float),
#                              f"{value_col}_sem": pd.Series([], dtype=float),
#                              "n": pd.Series([], dtype=int)})

#     g = dd.groupby(tile_col, observed=False)
#     res = g.agg(**{
#         f"{value_col}_mean": (value_col, "mean"),
#         f"{value_col}_sem":  (value_col, lambda x: x.std(ddof=1) / np.sqrt(max(1, x.count()))),
#         "n":                 (value_col, "count")
#     }).reset_index()

#     cats = dd[tile_col].cat.categories
#     res[tile_col] = pd.Categorical(res[tile_col], categories=cats, ordered=True)
#     res = res.set_index(tile_col).reindex(cats).reset_index()
#     res['n'] = res['n'].fillna(0).astype(int)
#     return res

# def _style(ax):
#     ax.grid(False)
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)

# def plot_tiles_line(tiles_ripple, tiles_barrage, value_col="isi_entropy",
#                     suptitle="ISI entropy vs ripple/barrage rate tiles"):
#     fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
#     y_label = "ISI entropy (bits)"

#     panels = [
#         (axes[0], tiles_ripple, "tile_ripple",  "Ripple rate quantiles"),
#         (axes[1], tiles_barrage,"tile_barrage", "Barrage rate quantiles")
#     ]
#     for ax, tiles, lab, title in panels:
#         if tiles is None or tiles.empty or (lab not in tiles.columns) or (f"{value_col}_mean" not in tiles.columns):
#             ax.text(0.5, 0.5, "No data", ha="center", va="center"); _style(ax); ax.set_title(title); continue
#         tp = tiles.dropna(subset=[f"{value_col}_mean"]).copy()
#         if tp.empty:
#             ax.text(0.5, 0.5, "No valid data", ha="center", va="center"); _style(ax); ax.set_title(title); continue

#         x = np.arange(len(tp))
#         y = tp[f"{value_col}_mean"].to_numpy(float)
#         yerr = tp[f"{value_col}_sem"].fillna(0).to_numpy(float)
#         ax.errorbar(x, y, yerr=yerr, fmt="-o", lw=1.8, ms=5, capsize=3, elinewidth=1.2)
#         xt = [f"{lvl} (n={int(n)})" for lvl, n in zip(tp[lab].astype(str), tp["n"].to_numpy())]
#         ax.set_xticks(x); ax.set_xticklabels(xt, rotation=30, ha='right')
#         ax.set_xlabel("Quantile tile"); ax.set_ylabel(y_label + " (mean ± SEM)")
#         ax.set_title(title); _style(ax)

#     fig.suptitle(suptitle, y=1.05); fig.tight_layout(rect=[0, 0.03, 1, 0.95])
#     return fig

# tiles_rip = aggregate_tiles(out_df, by="ripple",  value_col="isi_entropy", k=6)
# tiles_rip = tiles_rip.reset_index().rename(columns={'index': 'tile_ripple'})

# tiles_rip_d2 = aggregate_tiles(out_df, by="ripple",  value_col="d2", k=6)
# tiles_rip_d2 = tiles_rip_d2.reset_index().rename(columns={'index': 'tile_ripple'})
# tiles_barr= aggregate_tiles(out_df, by="barrage", value_col="isi_entropy", k=6)
# tiles_barr = tiles_barr.reset_index().rename(columns={'index': 'tile_barrage'})
# tiles_barr_d2= aggregate_tiles(out_df, by="barrage", value_col="d2", k=6)
# tiles_barr_d2 = tiles_barr_d2.reset_index().rename(columns={'index': 'tile_barrage'})
# save_path = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats\ripple_barr\New folder") / f"{animal_name}_{day}_isi_entropy_tiles.png"
# plt.savefig(save_path)
# plt.show()

# fig = plot_tiles_line(tiles_rip, tiles_barr, value_col="isi_entropy",
#                       suptitle="NREM ISI entropy vs ripple/barrage tiles")
# fig=plot_tiles_line(tiles_rip_d2, tiles_barr_d2, value_col="d2",
#                       suptitle="NREM D² vs ripple tiles")   
# save_path = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats\ripple_barr\New folder") / f"{animal_name}_{day}_d2_tiles.png"
# plt.savefig(save_path)
# plt.show()

In [388]:
# import pickle
# save_dir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats"  # <-- your target folder
# os.makedirs(save_dir, exist_ok=True)

# # build filename and save
# fname = f"{animal_name}_{day}.pkl"
# save_path = os.path.join(save_dir, fname)

# with open(save_path, "wb") as f:
#     pickle.dump({"per_center": out_df,
#                  "animal_name": animal_name,
#                  "day": day}, f)

# print(f"✅ Saved {save_path}")


In [389]:
# # Make 6-quantile tiles by ripple/barrage rate and plot
# tiles_rip = aggregate_tiles(out_df, by="ripple",  value_col="isi_entropy", k=6)
# tiles_rip = tiles_rip.reset_index().rename(columns={'index': 'tile_ripple'})

# tiles_rip_d2 = aggregate_tiles(out_df, by="ripple",  value_col="d2", k=6)
# tiles_rip_d2 = tiles_rip_d2.reset_index().rename(columns={'index': 'tile_ripple'})
# plt.show()

# fig = plot_tiles_line(tiles_rip, None, value_col="isi_entropy",
#                       suptitle="NREM ISI entropy vs ripple/barrage tiles")
# fig=plot_tiles_line(tiles_rip_d2, None, value_col="d2",
#                       suptitle="NREM D² vs ripple tiles")   

# plt.show()

NREM timecourse

In [390]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from scipy.stats import sem

# # ----------------------------------------------------
# # 1) Filter NREM + desired sleep epochs
# # ----------------------------------------------------
# df = ca1_complete_df.copy()

# # If your annotate function made column "sleep_epoch"
# # with values: "presleep", "midsleep", "postsleep"
# keep_epochs = ["presleep", "midsleep", "postsleep"]

# df_nrem = df[
#     (df["state"] == "NREM") &
#     (df["sleep_epoch"].isin(keep_epochs))
# ].copy()

# # Drop NaN d2
# df_nrem = df_nrem[df_nrem["d2"].notna()]


# # ----------------------------------------------------
# # 2) Bin time into 10-min windows
# # ----------------------------------------------------
# BIN = 600.0  # 10 minutes in seconds
# df_nrem["bin"] = (df_nrem["center_s"] // BIN).astype(int)


# # ----------------------------------------------------
# # 3) Compute mean and SEM per epoch × bin
# # ----------------------------------------------------
# grouped = (
#     df_nrem.groupby(["sleep_epoch", "bin"])
#     .agg(
#         d2_mean=("d2", "mean"),
#         d2_sem=("d2", sem),
#         t_center=("center_s", "mean")  # for plotting x-axis in seconds
#     )
#     .reset_index()
# )

# # Convert t_center into hours for nicer plotting
# grouped["t_hr"] = grouped["t_center"] / 3600.0


# # ----------------------------------------------------
# # 4) Plot timecourse: presleep / midsleep / postsleep
# # ----------------------------------------------------
# plt.figure(figsize=(10,5))

# epoch_colors = {
#     "presleep": "#1f77b4",
#     "midsleep": "#2ca02c",
#     "postsleep": "#d62728"
# }

# for ep in keep_epochs:
#     sub = grouped[grouped["sleep_epoch"] == ep]
#     plt.errorbar(
#         sub["t_hr"], sub["d2_mean"],
#         yerr=sub["d2_sem"],
#         marker="o", capsize=3,
#         label=ep,
#         color=epoch_colors.get(ep, None)
#     )

# plt.xlabel("Time (hr)")
# plt.ylabel("d²")
# plt.title("d² timecourse (10-min bins) across sleep epochs")
# plt.legend()
# plt.tight_layout()
# plt.show()
